# Week 5 — live coding (completed)

**Social and Cultural Analytics (7AAVBCS5)**

The filled-in version of what we typed in the lecture.

In [1]:
import subprocess
from pathlib import Path

import pandas as pd

# The module's data lives in a separate public repository so that this notebook
# works both on your own machine and on Colab.
DATA_REPO = "https://github.com/markjhill/7AAVBCS5-data.git"


def data_path(name: str) -> Path:
    """Return a usable path to a file or folder in the module's data folder.

    A notebook's "current folder" is wherever it was opened from, which differs
    between Positron and Colab and is the usual cause of FileNotFoundError. So
    rather than assuming, we search upwards for a `data` folder. If there isn't
    one -- which means we are on Colab -- the public data repository is
    downloaded once and used instead.

    Works for single files ("museum_data.csv") and for folders ("SUA").
    """
    here = Path.cwd()
    for candidate in (here, *here.parents):
        local = candidate / "data" / name
        if local.exists():
            return local

    downloaded = Path("7AAVBCS5-data")
    if not downloaded.is_dir():
        print("No local data folder found -- downloading the module data (once)...")
        subprocess.run(
            ["git", "clone", "--depth", "1", DATA_REPO, str(downloaded)],
            check=True,
        )
    target = downloaded / name
    if not target.exists():
        raise FileNotFoundError(f"{name!r} is not in the module data.")
    return target


papers = pd.read_csv(data_path("BritishAndIrishNewspapersTitleList_20191118.csv"),
                     low_memory=False)
print(papers.shape)

(24927, 24)


## Exercise 1: categorical data

In [2]:
# Which columns are text rather than numbers?
print(papers.dtypes.value_counts())
print()
print([c for c in papers.columns if papers[c].dtype == "object" or str(papers[c].dtype) == "str"][:8])

str        18
float64     5
int64       1
Name: count, dtype: int64

['publication_title', 'edition', 'preceding_titles', 'succeeding_titles', 'place_of_publication', 'country_of_publication', 'general_area_of_coverage', 'coverage_city']


In [3]:
country = papers["country_of_publication"].astype("category")
print(len(country.cat.categories), "categories")
print(country.value_counts(dropna=False).head(8).to_string())

40 categories
country_of_publication
England             20465
Scotland             1778
Ireland              1050
Wales                1019
Northern Ireland      415
England|Wales          58
Bermuda Islands        24
England|Scotland       13


Two things to notice immediately.

First, `place_of_publication` is missing for the great majority of titles:

In [4]:
print("place missing for", papers["place_of_publication"].isna().sum(),
      "of", len(papers), "titles",
      f"({papers['place_of_publication'].isna().mean() * 100:.1f}%)")

place missing for 20084 of 24927 titles (80.6%)


Second, some country values are **compounds** — `England|Wales` and 25 others:

In [5]:
compound = papers["country_of_publication"].dropna()
compound = compound[compound.str.contains("|", regex=False)]
print(compound.nunique(), "compound values covering", len(compound), "titles")
print(compound.value_counts().head(5).to_string())

26 compound values covering 146 titles
country_of_publication
England|Wales                                   58
England|Scotland                                13
England|Ireland                                 10
England|Ireland|Northern Ireland                10
England|United Kingdom Miscellaneous Islands    10


Is `England|Wales` one category or two? There is no right answer, only a documented one. We will
exclude them and say so.

In [6]:
main = papers[papers["country_of_publication"].isin(
    ["England", "Scotland", "Wales", "Ireland", "Northern Ireland"])].copy()
print(len(main), "titles in the five main countries")

24727 titles in the five main countries


## Exercise 2: tables and cross-tabulation

In [7]:
print(main["country_of_publication"].value_counts())

country_of_publication
England             20465
Scotland             1778
Ireland              1050
Wales                1019
Northern Ireland      415
Name: count, dtype: int64


`value_counts()` already sorts, most common first — R needed
`sort(table(x), decreasing = TRUE)`.

In [8]:
print("the mode:", main["country_of_publication"].value_counts().idxmax())
print("distinct countries:", main["country_of_publication"].nunique())

the mode: England
distinct countries: 5


In [9]:
print((main["country_of_publication"].value_counts(normalize=True) * 100).round(1))

country_of_publication
England             82.8
Scotland             7.2
Ireland              4.2
Wales                4.1
Northern Ireland     1.7
Name: proportion, dtype: float64


### Two-way: is digitisation even?

In [10]:
main["digitised"] = main["online_status"].notna()
pd.crosstab(main["country_of_publication"], main["digitised"], margins=True)

digitised,False,True,All
country_of_publication,,,
England,18840,1625,20465
Ireland,899,151,1050
Northern Ireland,367,48,415
Scotland,1593,185,1778
Wales,962,57,1019
All,22661,2066,24727


Read the counts and ask what they show. The obvious answer is "England dominates" — 1,625 digitised
titles against Wales's 57.

That is almost content-free, because England has twenty times as many titles to begin with. The
table is mostly measuring the size of England.

In [11]:
(pd.crosstab(main["country_of_publication"], main["digitised"],
             normalize="index") * 100).round(1)

digitised,False,True
country_of_publication,,
England,92.1,7.9
Ireland,85.6,14.4
Northern Ireland,88.4,11.6
Scotland,89.6,10.4
Wales,94.4,5.6


As a share of each country's **own** holdings, the order reverses:

| | % digitised |
|---|---|
| **Ireland** | **14.4** |
| Northern Ireland | 11.6 |
| Scotland | 10.4 |
| England | 7.9 |
| **Wales** | **5.6** |

Ireland is the most digitised; Wales the least; England fourth of five. The overall rate is 8.4%.

Both tables are true. The counts measure **volume**, the percentages measure **rate**, and only the
second answers "has digitisation been even?"

In [12]:
# The other margin answers a different question again.
(pd.crosstab(main["country_of_publication"], main["digitised"],
             normalize="columns") * 100).round(1)

digitised,False,True
country_of_publication,,
England,83.1,78.7
Ireland,4.0,7.3
Northern Ireland,1.6,2.3
Scotland,7.0,9.0
Wales,4.2,2.8


- "What share of **Welsh titles** are digitised?" → 5.6%
- "What share of **digitised titles** are Welsh?" → 2.8%

Confusing those two is the commonest way a crosstab gets misread.

In [13]:
# Tables are data frames.
counts = main["country_of_publication"].value_counts().reset_index()
counts.columns = ["country", "titles"]
counts

,country,titles
0,England,20465
1,Scotland,1778
2,Ireland,1050
3,Wales,1019
4,Northern Ireland,415


## Exercise 3: grouping

In [14]:
main.groupby("country_of_publication")["first_date_held"].agg(["count", "median"])

,count,median
country_of_publication,,
England,20203,1926.0
Ireland,1036,1867.0
Northern Ireland,406,1911.0
Scotland,1710,1896.0
Wales,1000,1913.0


Ireland's holdings begin earliest — median 1874 against England's 1935.

Always keep `count`. A median over a handful of titles is not a finding.

In [15]:
main.groupby(["country_of_publication", "digitised"])["first_date_held"].agg(["count", "median"])

count  median
country_of_publication digitised               
England                False      18590  1935.0
                       True        1613  1852.0
Ireland                False        885  1874.0
                       True         151  1848.0
Northern Ireland       False        358  1935.0
                       True          48  1853.0
Scotland               False       1526  1899.0
                       True         184  1865.5
Wales                  False        943  1918.0
                       True          57  1869.0

### The finding

**Digitised titles are older, in every single country:**

| Country | digitised | not digitised | gap |
|---|---|---|---|
| England | 1852 | 1935 | 83 years |
| Ireland | 1848 | 1874 | 26 |
| Northern Ireland | 1853 | 1935 | 82 |
| Scotland | 1865.5 | 1899 | 33.5 |
| Wales | 1869 | 1918 | 49 |

No exceptions, which makes it far more convincing than any single-country result.

Hypotheses, not conclusions: copyright expiry, established scholarly demand for the
nineteenth-century press, preservation arguments for fragile paper, and the historical scoping of
the BNA and BURNEY collections.

The methodological point is the one that matters. **Study "British newspapers" using only digitised
material and you are studying a sample that is systematically older than the population.** That
selection effect is built into the infrastructure, not into your code — and no amount of careful
analysis downstream will remove it.

This is chapter §5.4: the interaction between the spatial, temporal and institutional dimensions is
the finding, not a footnote to it.

## What to take away

| Dimensions | Tool |
|---|---|
| one category | `value_counts()` |
| two categories | `pd.crosstab()` |
| category and a number | `groupby().agg()` |

And: **always ask whether you want counts or percentages, and which margin.** The answer follows
from your question, not from preference.

End of live coding.